# S7.1 · 隐私攻击与防护评估

攻击者设定为**诚实但好奇的协议内参与方**，不假设外部窃听。

| 编号 | 攻击 | 谁攻击谁 | 暴露面 |
|---|---|---|---|
| A1 | 标签推断 | 被动方 → 主动方标签 | 每轮下发的残差 |
| A2 | 嵌入反演 | 主动方 → 被动方特征 | 上传的嵌入 |
| A3 | 梯度标签推断 | 被动方 → 主动方标签 | 回传的梯度（仅形态A） |

In [1]:
ROUND_DP = 4          # 表格展示精度（不影响任何计算结果）
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "registry").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml
CONFIG_PATH = ROOT / "modules/m5_modeling/configs/experiment.yaml"
config = yaml.safe_load(open(CONFIG_PATH, encoding="utf-8"))
seed = config.get("seeds", [config.get("seed")])[0]
git = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, cwd=ROOT).stdout.strip()
print("config:", CONFIG_PATH.relative_to(ROOT))
print("seed  :", seed, "| 全部种子:", config.get("seeds"))
print("git   :", git or "(未提交)")
print("numpy :", np.__version__, "| pandas:", pd.__version__)

config: modules/m5_modeling/configs/experiment.yaml
seed  : 11 | 全部种子: [11, 22, 33, 44, 55]
git   : 7f29b9e
numpy : 2.3.5 | pandas: 2.3.3


In [2]:
atk = pd.read_csv(ROOT / 'modules/m7_security/results/attack_results.csv')
a1 = atk[atk.attack == 'A1_残差标签推断'].groupby('dp_sigma')[
    ['eps_per_round','utility_auc','leak_auc_首轮','leak_auc_最优轮']].mean()
a1.round(ROUND_DP)

,eps_per_round,utility_auc,leak_auc_首轮,leak_auc_最优轮
dp_sigma,,,,
0.00,inf,0.7868,1.0000,1.0000
0.01,484.4805,0.7869,1.0000,1.0000
0.03,161.4935,0.7869,1.0000,1.0000
0.10,48.4481,0.7870,1.0000,1.0000
0.30,16.1494,0.7872,0.9878,0.9952
1.00,4.8448,0.7869,0.7574,0.7981
3.00,1.6149,0.7771,0.5933,0.6481
10.00,0.4845,0.7109,0.5291,0.5892


首轮泄露 AUC = 1.0000：训练开始时权重为零、预测恒为 0.5，残差 `r = 0.5 − y` 的符号与标签**一一对应**。**不加防护的纵向联邦逻辑回归，标签是完全泄露的。**

## 证伪检验：跨轮平均攻击

上表看起来 σ=1.0 就能把泄露压到 0.76 而几乎不损失可用性——这个结论**太好了**。

但攻击者只用了单轮残差。噪声在轮间独立、标签恒定，**跨轮平均即可把噪声消掉**。这是必须自己打的证伪。

In [3]:
mr = pd.read_csv(ROOT / 'modules/m7_security/results/multiround_attack.csv')
mr.groupby('dp_sigma')[['可用性AUC','单轮攻击','跨轮平均攻击','前50轮平均']].mean().round(ROUND_DP)

,可用性AUC,单轮攻击,跨轮平均攻击,前50轮平均
dp_sigma,,,,
0.0,0.7868,1.0000,1.0000,1.0000
0.3,0.7872,0.9878,1.0000,1.0000
1.0,0.7869,0.7574,1.0000,1.0000
3.0,0.7771,0.5933,0.9999,0.9446
10.0,0.7109,0.5291,0.9037,0.6851
30.0,0.6352,0.5149,0.6659,0.5659


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "Heiti TC", "PingFang SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
FIGSIZE_WIDE = (9, 4)
FIG_DPI = 150
GRID_W = 11
GRID_H = 7
FIGDIR = ROOT / "modules/m7_security" / "results"
m = mr.groupby('dp_sigma')[['可用性AUC','单轮攻击','跨轮平均攻击']].mean()
L0_REF = 0.7089
fig, ax = plt.subplots(figsize=FIGSIZE_WIDE)
ax.plot(m.index, m['单轮攻击'], marker='o', label='泄露 AUC（单轮攻击）')
ax.plot(m.index, m['跨轮平均攻击'], marker='s', label='泄露 AUC（跨轮平均攻击）')
ax.plot(m.index, m['可用性AUC'], marker='^', label='模型可用性 AUC')
ax.axhline(L0_REF, color='red', linestyle='--', label='L0 内地单方基线')
ax.set_xscale('symlog'); ax.set_xlabel('高斯噪声 σ'); ax.set_ylabel('AUC')
ax.set_title('逐轮加噪防护：跨轮平均攻击下失效')
ax.legend(); fig.tight_layout()
fig.savefig(FIGDIR / 'dp_privacy_utility.png', dpi=FIG_DPI); plt.close(fig)
print('图已保存 dp_privacy_utility.png')

图已保存 dp_privacy_utility.png


**结论（推翻了上一节的乐观读数）**：

- σ=1.0 时跨轮平均攻击的泄露 AUC 回到 **1.0000**
- 要把泄露压到 0.67，需要 σ=30，此时可用性降至 0.6352，**低于 L0 内地单方基线 0.7089**

→ **逐轮加高斯噪声不是本协议的有效防护**。把泄露压下去所需的噪声，会先把联邦模型的价值清零。有效路径只能是**协议级**手段（安全聚合 / 同态加密 / 秘密分享）或改变暴露面。

## A2 嵌入反演 / A3 梯度标签推断

In [5]:
a2 = atk[atk.attack == 'A2_嵌入反演'].groupby('protocol')[
    ['utility_auc','inv_r2_mean','inv_r2_max','leak_auc_梯度方向']].mean()
a2.round(ROUND_DP)

,utility_auc,inv_r2_mean,inv_r2_max,leak_auc_梯度方向
protocol,,,,
L3c_形态A_双向,0.7690,0.4877,0.7497,1.0
L3c_形态B_自监督,0.7402,0.6666,0.8745,0.5
L3c_形态B_随机,0.7267,0.4964,0.7592,0.5


两个反直觉的结果：

1. **形态B 挡住了梯度标签泄露**（A3 从 1.00 降到 0.50）——不回传梯度，暴露面直接消失。
2. **形态B 并没有降低特征反演风险，反而更糟**（自监督形态 R²=0.667 > 形态A 的 0.488）。PCA 编码器是线性的、更容易求逆；随标签训练的编码器反而丢掉了更多与任务无关的信息。

→ **「冻结编码器 = 更安全」是错的**。它换掉的是标签暴露面，不是特征暴露面。